# NHANES 2021-2023 — Health Data Analysis

**Domande di ricerca:** prevedere diabete e pressione alta da dati di questionario; capire i driver del BMI.

**Dataset:** NHANES (CDC) ciclo 2021-2023 — 11.933 partecipanti × 1.112 variabili.

Ogni sezione del notebook espone prima il **ragionamento**, poi il codice.

## 1. Caricamento dati + dizionario

**Ragionamento:** con 1.112 colonne, il dizionario (`DATA_DICTIONARY.csv`) è la mappa: lo carichiamo PRIMA dei dati e lo usiamo per capire cosa esiste. Mai esplorare alla cieca.

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

HANDLE = "rileyzurrin/nhanes-2021-2023-public-data-and-codebooks"

# Dizionario: codice colonna -> etichetta umana
df = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, HANDLE, "DATA_DICTIONARY.csv")

# Dati veri
df_participants = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, HANDLE, "data/participants.csv")

df_participants.shape  # (11933, 1112)

## 2. Rinomina colonne: dai codici CDC alle etichette umane

**Ragionamento:** le colonne si chiamano `DIQ010`, `BPQ020`... illeggibili. Il dizionario contiene la colonna `label` con il testo della domanda. Costruiamo una mappa `codice -> etichetta` con `dict(zip(...))` e rinominiamo tutto in un colpo solo.

**Attenzione:** il filtro deve essere esatto — `output_file == "data/participants.csv"`, non un nome inventato a memoria.

In [ ]:
diz_part = df[df["output_file"] == "data/participants.csv"]
mappa = dict(zip(diz_part["output_column"], diz_part["label"]))

df_clean = df_participants.rename(columns=mappa)
print(df_clean.shape)
df_clean.columns[:10]

## 3. Caccia alle colonne: il filtro trova, l'analyst decide

**Ragionamento:** cerchiamo per concetto nelle etichette. Il filtro però è ingenuo: cerca le parole, non il significato. Esempio reale: cercando "physical activities" esce anche `"Leak urine during physical activities?"` — parola giusta, contesto sbagliato. **Si scarta.**

In [ ]:
for parola in ["diabetes", "blood pressure", "Body Mass", "smoke", "physically active"]:
    risultati = [c for c in df_clean.columns if parola.lower() in str(c).lower()]
    print(f"--- {parola.upper()} ---")
    for r in risultati:
        print(" ", r)

## 4. Selezione colonne e subset

**Ragionamento:**
- 3 target: diabete, pressione alta, BMI
- Feature: età (fondamentale per entrambe le condizioni), genere, fumo, sedentarietà
- `Minutes sedentary activity` preferita a `Days physically active at least 60 min.` (domanda fatta solo ai ragazzi: 3.041 risposte su 11.933)
- Colonne "conseguenza della diagnosi" (età alla diagnosi, farmaci): tenute per l'EDA, escluse poi dal modello (leakage)

In [ ]:
colonne_scelte = [
    # TARGET
    "Doctor told you have diabetes",
    "Ever told you had high blood pressure",
    "Body Mass Index (kg/m**2)",
    # FEATURE
    "Age in years at screening",
    "Gender",
    "Smoked at least 100 cigarettes in life",
    "Minutes sedentary activity",
    "Ever told you have prediabetes",
    # Solo per EDA (leakage se usate come feature)
    "Age when first told you had diabetes",
    "Told had high blood pressure - 2+ times",
    "Taking high blood pressure medication",
]

df_sub = df_clean[colonne_scelte]
df_sub.info()

**Leggere l'info():** i Non-Null Count raccontano gli *skip pattern* del questionario — "Age when first told..." ha 1.081 risposte perché la fanno solo ai diabetici. Quei NaN = "domanda non applicabile", non dati persi.

## 5. Ricodifica dei codici speciali NHANES

**Ragionamento:** NHANES codifica 1=Yes, 2=No, ma anche 3=Borderline, 7=Refused, 9=Don't know. Lo standard ML è 1/0. Il trucco: `map({1:1, 2:0})` — i valori non presenti nel dizionario (3, 7, 9) diventano **automaticamente NaN**. Poi `dropna(subset=...)` solo sui target: mai sull'intero dataset (su NHANES restituirebbe 0 righe!).

In [ ]:
mappa_binaria = {1: 1, 2: 0}
colonne_si_no = ["Doctor told you have diabetes",
                 "Ever told you had high blood pressure",
                 "Smoked at least 100 cigarettes in life",
                 "Ever told you have prediabetes"]

for col in colonne_si_no:
    df_sub[col] = df_sub[col].map(mappa_binaria)

df_sub = df_sub.dropna(subset=["Doctor told you have diabetes",
                               "Ever told you had high blood pressure",
                               "Body Mass Index (kg/m**2)"])
df_sub.shape  # (6287, 12)

**Perché 6.287 righe e non 11.933?** Il BMI viene *misurato* fisicamente in ambulatorio mobile, le domande invece le fanno a quasi tutti. Il dropna sui 3 target tiene solo chi ha completato sia questionario che esame. 6.287 righe restano più che sufficienti.

## 6. Class imbalance

**Ragionamento:** contiamo le classi PRIMA di modellare.

In [ ]:
print("DIABETE:")
print(df_sub["Doctor told you have diabetes"].value_counts())
print("\nPRESSIONE ALTA:")
print(df_sub["Ever told you had high blood pressure"].value_counts())

- **Diabete ~1:10** → paradosso dell'accuratezza: un modello che dice sempre "No" farebbe 90% trovando zero diabetici. Serviranno `stratify` + `class_weight="balanced"`.
- **Pressione ~1:1.9** (34% positivi, plausibile: l'ipertensione adulta USA è su quei livelli) → sbilanciamento lieve, gestibile senza pesi.

## 7. Describe e il codice 9999

**Ragionamento:** nei dati numerici i min/max sospetti tradiscono i codici speciali.

In [ ]:
df_sub[["Age in years at screening", "Body Mass Index (kg/m**2)", "Minutes sedentary activity"]].describe()

Il max di "Minutes sedentary" è **9999 minuti = 6,9 giorni seduti di fila**: impossibile, è il codice NHANES per "non risposto". BMI max 74.8 invece è estremo ma **reale** (obesità severa) e si tiene.

In [ ]:
df_sub["Minutes sedentary activity"] = df_sub["Minutes sedentary activity"].replace(9999, np.nan)
df_sub["Minutes sedentary activity"].describe()

## 8. Matrice di correlazione

**Ragionamento:** la heatmap su TUTTE le colonne (non solo le numeriche continue): ciò che interessa è la relazione **feature → target**, che una matrice 3×3 non mostrerebbe.

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df_sub.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlazioni — feature e target")
plt.show()

**Lettura:** età → pressione 0.43 (la più forte), età → diabete 0.26, prediabete → diabete 0.31, BMI → pressione 0.23. Sedentarietà e fumo quasi zero *linearmente* — ma vedremo che il RandomForest la pensa diversamente (i alberi trovano soglie, non solo rette).

## 9. X, y e il target leakage

**Ragionamento:** fuori da X tutti i target E le colonne-conseguenza della diagnosi. "Age when first told you had diabetes" esiste solo per chi HA il diabete: esistere = risposta. "Prediabetes" resta: è una diagnosi *precedente*, informazione legittima.

I NaN residui nelle feature si riempiono con la **mediana** (robusta agli outlier).

In [ ]:
from sklearn.impute import SimpleImputer

y_diabete = df_sub["Doctor told you have diabetes"]
y_pressione = df_sub["Ever told you had high blood pressure"]

colonne_da_togliere = [
    # i tre TARGET (mai dentro X: sarebbero le risposte nel compito!)
    "Doctor told you have diabetes",
    "Ever told you had high blood pressure",
    "Body Mass Index (kg/m**2)",
    # LEAKAGE diabete: esiste solo per chi HA già il diabete
    "Age when first told you had diabetes",
    # LEAKAGE pressione: conseguenze della diagnosi, non cause
    "Told had high blood pressure - 2+ times",
    "Taking high blood pressure medication",
]

X = df_sub.drop(columns=colonne_da_togliere)
imputer = SimpleImputer(strategy="median")
X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
X.columns

## 10. Modello 1 — Diabete

**Ragionamento:** `stratify=y` mantiene il rapporto 1:10 in train e test; `class_weight="balanced"` dice al modello che sbagliare un diabetico costa ~10 volte di più.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

X_train, X_test, y_train, y_test = train_test_split(
    X, y_diabete, test_size=0.2, random_state=42, stratify=y_diabete)

rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["No diabete", "Diabete"]))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=["No", "Si"]).plot()
plt.show()

**Risultato:** accuracy 0.72 (sotto il 90% del modello-furbo "sempre No", *di proposito*), recall diabete 0.42 → 71 diabetici trovati su 169, al prezzo di 259 falsi allarmi. Trade-off tipico da screening.

### Feature importance

In [ ]:
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
imp.plot(kind="barh", title="Cosa predice il diabete?")
plt.show()

Età dominante (~63%), poi sedentarietà (~21%) — che nella correlazione *lineare* valeva ~0. Lezione: heatmap e RandomForest vedono cose diverse, si usano insieme.

## 11. Modello 2 — Pressione alta

**Ragionamento:** X è già pronto, cambia solo la y. Sbilanciamento lieve → niente `class_weight`. **Nomi variabili nuovi** (`X2_...`): riusare `x_train` del modello precedente mischia righe e etichette senza nessun errore visibile (bug reale capitato in questo progetto: typo `x_tain`).

In [ ]:
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X, y_pressione, test_size=0.2, random_state=42, stratify=y_pressione)

rf2 = RandomForestClassifier(n_estimators=200, random_state=42)
rf2.fit(X2_train, y2_train)

y2_pred = rf2.predict(X2_test)
print("Accuracy:", accuracy_score(y2_test, y2_pred))
print(classification_report(y2_test, y2_pred, target_names=["No ipertensione", "Ipertensione"]))
ConfusionMatrixDisplay(confusion_matrix(y2_test, y2_pred), display_labels=["No", "Si"]).plot()
plt.show()

**Risultato:** accuracy 0.65, recall ipertensione 0.46.

## 12. Conclusioni e limiti

1. **L'età è il driver principale** sia per diabete che per pressione alta — confermato da correlazione E feature importance.
2. **Recall modesti (0.42-0.46) perché le feature sono solo da questionario.** Gli esami del sangue (HbA1c, glicemia) esistono nel dataset ma sono stati esclusi *volontariamente*: prevedere "il dottore ti ha detto che hai il diabete" usando l'esame che *definisce* il diabete sarebbe leakage. Un 95% così sarebbe finto.
3. **Metodologia difendibile:** gestione skip pattern, codici speciali (3/7/9/9999), leakage evitato, imbalance gestito con stratify + class_weight, valutazione oltre l'accuracy.
4. **Bug memorabile:** il typo `x_tain` → allenamento su dati misallineati, accuracy falsata 0.59 → 0.65 dopo il fix. In Jupyter le variabili vecchie restano in memoria: nomi distinti per modelli distinti.